Problem 1 – Row-wise ℓ₂ normalization
-------------------------------------
Given a float tensor **X** of shape `(B, D)`, return a tensor **Y**
with the same shape such that every row of **Y** has (approximate)
ℓ₂-norm == 1.

Mathematically  
 Yᵢ = Xᵢ / (‖Xᵢ‖₂ + ε)

Constraints
-----------
* 1 ≤ B·D ≤ 10⁷
* Use **no Python for-loops**.

Example
-------
```python
X = torch.tensor([[3., 4.]])
Y = normalize_rows(X)
Y.norm(dim=1)

>>> tensor([1.])
```

In [4]:
import torch

def normalize_rows(X: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    return X / (torch.linalg.vector_norm(X, ord=2, dim=-1, keepdim=True) + eps)


# ---------- reference + tests (do not modify) ----------
def _ref_normalize_rows(X, eps=1e-8):
    return X / (X.norm(dim=1, keepdim=True) + eps)

def _self_check():
    X = torch.randn(5, 7)
    try:
        Y = normalize_rows(X)
    except NotImplementedError:
        print("🔧  Implement normalize_rows and re-run the cell.")
        return
    assert Y.shape == X.shape, "Shape mismatch"
    assert torch.allclose(Y, _ref_normalize_rows(X)), "Wrong answer"
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 2 – Channel-wise z-score
--------------------------------
For an image batch tensor **imgs** of shape `(N, C, H, W)` compute

    (img − μ_c) / (σ_c + ε)

where μ_c and σ_c are the mean and std of channel *c* over the
(N, H, W) axes.

Return a tensor of the same shape.

No Python loops allowed.

In [10]:
import torch

def channel_zscore(imgs: torch.Tensor, eps: float = 1e-5) -> torch.Tensor:
    N, C, H, W = imgs.shape
    tmp_imgs = imgs.permute(1, 0, 2, 3).reshape(C, N*H*W)
    avg = tmp_imgs.mean(dim=-1).reshape(1, C, 1, 1)
    std = tmp_imgs.std(dim=-1).reshape(1, C, 1, 1)
    return (imgs - avg) / (std + eps)

# reference & test
def _ref_channel_zscore(imgs, eps=1e-5):
    mean = imgs.mean(dim=(0, 2, 3), keepdim=True)
    std  = imgs.std (dim=(0, 2, 3), keepdim=True) + eps
    return (imgs - mean) / std

def _self_check():
    imgs = torch.randn(4, 3, 16, 16)
    try:
        out = channel_zscore(imgs)
    except NotImplementedError:
        print("🔧  Implement channel_zscore.")
        return
    assert torch.allclose(out, _ref_channel_zscore(imgs), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 3 – Numerically stable softmax per row
----------------------------------------------
Given a tensor **S** of shape `(B, K)`, return

    softmax(Sᵢ)  for each row i,

implemented with the usual “shift by max” trick to avoid overflow.

In [13]:
import torch

def stable_row_softmax(S: torch.Tensor) -> torch.Tensor:
    B, K = S.shape
    S = S - S.max(dim=-1, keepdim=True).values
    return S.exp() / S.exp().sum(dim=-1, keepdim=True)


def _ref_stable_row_softmax(S):
    Z = S - S.max(dim=1, keepdim=True).values
    eZ = Z.exp()
    return eZ / eZ.sum(dim=1, keepdim=True)

def _self_check():
    S = torch.randn(3, 9) * 25
    try:
        out = stable_row_softmax(S)
    except NotImplementedError:
        print("🔧  Implement stable_row_softmax.")
        return
    assert torch.allclose(out, _ref_stable_row_softmax(S), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 4 – Masked fill
-----------------------
Given
* **logits** of arbitrary shape `(*)`
* a boolean **mask** of the same shape  
return a tensor where entries with `mask == False` are replaced by
−∞ (so they will become zero after softmax).

In [14]:
import torch

def masked_logits(logits: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    return logits.masked_fill_(~mask, -torch.inf)


def _ref_masked_logits(logits, mask):
    return logits.masked_fill(~mask, float("-inf"))

def _self_check():
    L = torch.randn(2, 5)
    M = torch.rand(2, 5) > .4
    try:
        out = masked_logits(L, M)
    except NotImplementedError:
        print("🔧  Implement masked_logits.")
        return
    assert torch.isneginf(out[~M]).all()
    assert torch.equal(out, _ref_masked_logits(L, M))
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 5 – Batched cosine
--------------------------
Input
* **A** ∈ ℝ^{B×M×D}
* **B** ∈ ℝ^{B×N×D}

Output **S** ∈ ℝ^{B×M×N} where  
 S[b, i, j] = cosine_similarity(A[b,i], B[b,j]).

In [15]:
import torch

def batched_cosine(A: torch.Tensor, B: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    b, m, d = A.shape
    b, n, d = B.shape

    P = torch.einsum("bmd,bnd->bmn", A, B)
    normA = torch.linalg.vector_norm(A, dim=-1, keepdim=True)
    normB = torch.linalg.vector_norm(B, dim=-1, keepdim=True).view(b, 1, n)

    return P / (normA * normB + eps)


def _ref_batched_cosine(A, B, eps=1e-8):
    A = A / (A.norm(dim=-1, keepdim=True) + eps)
    B = B / (B.norm(dim=-1, keepdim=True) + eps)
    return torch.einsum("bmd,bnd->bmn", A, B)

def _self_check():
    A, B = torch.randn(2, 3, 4), torch.randn(2, 5, 4)
    try:
        out = batched_cosine(A, B)
    except NotImplementedError:
        print("🔧  Implement batched_cosine.")
        return
    assert torch.allclose(out, _ref_batched_cosine(A, B), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 6 – One-hot
-------------------
Given an integer index tensor **idx** of shape `(B,)` and
num_classes =C, return a one-hot tensor `(B, C)` without any loop.

In [17]:
import torch

def one_hot(idx: torch.Tensor, num_classes: int) -> torch.Tensor:
    B, = idx.shape
    idx = idx.view(B, 1)
    val = torch.ones(B, 1)
    res = torch.zeros(B, num_classes)
    res.scatter_(1, idx, val)
    return res


def _ref_one_hot(idx, C):
    out = torch.zeros(idx.size(0), C, device=idx.device)
    out.scatter_(1, idx.unsqueeze(1), 1.0)
    return out

def _self_check():
    i = torch.tensor([2, 0, 1])
    try:
        out = one_hot(i, 4)
    except NotImplementedError:
        print("🔧  Implement one_hot.")
        return
    assert torch.equal(out, _ref_one_hot(i, 4))
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 7 – Masked cumulative product
-------------------------------------
Input
* **X** ∈ ℝ^{B×T} (positive)
* **reset** ∈ {0,1}^{B×T}

For each row, compute the running product but **restart at 1** whenever
reset==1 at that timestep.

In [21]:
import torch

def masked_cumprod(X: torch.Tensor, reset: torch.Tensor) -> torch.Tensor:
    assert torch.all(X != 0.0), "assumption voilated"
    
    P = torch.cumprod(X, dim=1)
    PP = torch.cat((torch.ones_like(P[:, :1]), P[:, :-1]), dim=1)

    index = reset.cumsum(dim=1)
    tmp = torch.zeros_like(X)
    tmp.scatter_reduce_(1, index, reset * PP, reduce="sum")
    tmp[:, 0] = 1

    baseline = torch.gather(tmp, 1, index)
    return P / baseline

def _ref_masked_cumprod(X, reset):
    B, T = X.shape
    running = torch.ones(B)
    res = torch.ones(B, T)
    for t in range(T):
        running = torch.where(reset[:, t], X[:, t], running * X[:, t])
        res[:, t] = running
    return res
    
def _self_check():
    X = torch.tensor([[2.,3.,4.,5.],[1.,2.,3.,4.]])
    R = torch.tensor([[0,1,0,0],[0,0,1,0]], dtype=torch.bool)
    try:
        out = masked_cumprod(X, R)
    except NotImplementedError:
        print("🔧  Implement masked_cumprod.")
        return
    assert torch.allclose(out, _ref_masked_cumprod(X, R))
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 8 – Conv2d (no bias) with unfold
----------------------------------------
Re-implement `torch.nn.functional.conv2d` for square kernels
using `torch.nn.functional.unfold`.

Signature
---------
conv2d_im2col(x, w, stride=1, padding=0)

In [1]:
import torch

def conv2d_im2col(x: torch.Tensor, w: torch.Tensor,
                  stride: int = 1, padding: int = 0) -> torch.Tensor:
    B, Ci, H, W = x.shape
    Co, _, k, _ = w.shape
    x_unf = torch.nn.functional.unfold(x, k, padding=padding, stride=stride) # B x (Ci * k * k) x L
    w = w.view(Co, Ci * k * k)
    out = torch.einsum("bdl,cd->bcl", x_unf, w)
    Ho = (H + 2*padding - k) // stride + 1
    Wo = (W + 2*padding - k) // stride + 1
    return out.view(B, Co, Ho, Wo)

def _ref_conv2d_im2col(x, w, stride=1, padding=0):
    N, C_in, H, W = x.shape
    C_out, _, k, _ = w.shape
    x_unf = torch.nn.functional.unfold(x, k, padding=padding, stride=stride)
    out = w.view(C_out, -1) @ x_unf
    H_out = (H + 2*padding - k)//stride + 1
    W_out = (W + 2*padding - k)//stride + 1
    return out.view(N, C_out, H_out, W_out)

def _self_check():
    x = torch.randn(1,3,5,5); w = torch.randn(4,3,3,3)
    try:
        out = conv2d_im2col(x, w, padding=1)
    except NotImplementedError:
        print("🔧  Implement conv2d_im2col.")
        return
    assert torch.allclose(out, _ref_conv2d_im2col(x, w, padding=1), atol=1e-5)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 9 – Batched k-NN (squared Euclidean)
--------------------------------------------
Given X ∈ ℝ^{B×N×D}, Y ∈ ℝ^{B×M×D} and k, return indices of the k
closest Y-vectors for every X-vector within each batch.

In [5]:
import torch

def batched_knn(X: torch.Tensor, Y: torch.Tensor, k: int) -> torch.Tensor:
    B, N, D = X.shape
    B, M, D = Y.shape

    xx = (X * X).sum(dim=-1, keepdim=True)                  # B, N, 1
    yy = (Y * Y).sum(dim=-1, keepdim=True).permute(0, 2, 1) # B, 1, M
    xy = torch.einsum("bnd,bmd->bnm", X, Y)                 # B, N, M

    l2 = xx + yy - 2*xy                                     # B, N, M
    _, idx = l2.topk(k, dim=-1, largest=False)
    return idx


def _ref_batched_knn(X, Y, k):
    d = (X.pow(2).sum(-1, keepdim=True)
         + Y.pow(2).sum(-1).unsqueeze(1)
         - 2*torch.einsum("bnd,bmd->bnm", X, Y))
    _, idx = d.topk(k, dim=-1, largest=False)
    return idx

def _self_check():
    X,Y = torch.randn(2,4,6), torch.randn(2,5,6)
    try:
        idx = batched_knn(X, Y, 3)
    except NotImplementedError:
        print("🔧  Implement batched_knn.")
        return
    assert torch.equal(idx, _ref_batched_knn(X,Y,3))
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 10 – Brevity penalty
----------------------------
For sentence lengths `len_c, len_r` (candidate vs reference),
return

    BP = 1                   if len_c > len_r  
         exp(1 − len_r/len_c) otherwise

Vectorized over a batch of shape `(B,)`.

In [6]:
import torch
import math

def brevity_penalty(len_c: torch.Tensor, len_r: torch.Tensor) -> torch.Tensor:
    len_c, len_r = len_c.float(), len_r.float()
    return torch.where(len_c > len_r, torch.ones_like(len_c), torch.exp(1 - len_r / (len_c + 1e-8)))


def _ref_bp(lc, lr):
    lc, lr = lc.float(), lr.float()
    return torch.where(lc > lr, torch.ones_like(lc), torch.exp(1 - lr / (lc + 1e-8)))

def _self_check():
    lc, lr = torch.tensor([10, 5]), torch.tensor([8, 9])
    try:
        bp = brevity_penalty(lc, lr)
    except NotImplementedError:
        print("🔧  Implement brevity_penalty.")
        return
    assert torch.allclose(bp, _ref_bp(lc, lr))
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 11 – One-step GRU
-------------------------
The GRU update consists of the following steps:

1. Reset gate:
    r = sigmoid(Wx_r @ x + Wh_r @ h + b_r)

2. Update gate:
    z = sigmoid(Wx_z @ x + Wh_z @ h + b_z)

3. Candidate (new) state:
    n = tanh(Wx_n @ x + r ⊙ (Wh_n @ h) + b_n)

4. Final hidden state:
    h_next = (1 - z) ⊙ h + z ⊙ n

In [11]:
import torch

def gru_cell(x: torch.Tensor, h: torch.Tensor,
             Wx: torch.Tensor, Wh: torch.Tensor,
             b: torch.Tensor) -> torch.Tensor:
    #  x:  B, D
    #  h:  B, H
    # Wx: 3H, D
    # Wh: 3H, H
    #  b: 3H
    B, D = x.shape
    _, H = h.shape

    x_proj = x @ Wx.T
    h_proj = h @ Wh.T
    b_proj = b

    x_r, x_z, x_n = x_proj.chunk(3, dim=-1)
    h_r, h_z, h_n = h_proj.chunk(3, dim=-1)
    b_r, b_z, b_n = b_proj.chunk(3, dim=-1)

    r = torch.sigmoid(x_r + h_r + b_r)
    z = torch.sigmoid(x_z + h_z + b_z)
    n = torch.tanh(x_n + r * h_n + b_n)

    return (1 - z) * h + z * n
    


def _ref_gru_cell(x,h,Wx,Wh,b):
    H = h.size(1)

    # Input and hidden projections for all 3 gates
    x_proj = x @ Wx.T + b      # (B, 3H)
    h_proj = h @ Wh.T          # (B, 3H)

    # Split into reset, update, and new gate
    x_r, x_z, x_n = x_proj.chunk(3, dim=-1)
    h_r, h_z, h_n = h_proj.chunk(3, dim=-1)

    # Gates
    r = torch.sigmoid(x_r + h_r)             # Reset gate
    z = torch.sigmoid(x_z + h_z)             # Update gate
    n = torch.tanh(x_n + r * h_n)            # New gate with r ⊙ h_n

    # Final hidden state
    h_next = (1 - z) * h + z * n             # GRU interpolation
    return h_next

def _self_check():
    B,D,H=2,3,4
    args=[torch.randn(B,D),torch.randn(B,H),
          torch.randn(3*H,D),torch.randn(3*H,H),
          torch.randn(3*H)]
    try:
        out=gru_cell(*args)
    except NotImplementedError:
        print("🔧  Implement gru_cell.")
        return
    assert torch.allclose(out,_ref_gru_cell(*args),atol=1e-5)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 12 – Causal self-attention scores
-----------------------------------------
Given Q,K ∈ ℝ^{B×T×d}, return the softmaxed score matrix
`(B, T, T)` with an upper-triangular causal mask.

In [20]:
import torch, math

def causal_attention_scores(Q: torch.Tensor, K: torch.Tensor) -> torch.Tensor:
    # Q: B, T, D
    # K: B, S, D
    B, T, D = Q.shape
    _, S, _ = K.shape
    
    mask = torch.tril(torch.ones(T, S, dtype=torch.bool), diagonal=0)
    
    dot_products = torch.einsum("btd,bsd->bts", Q, K) / math.sqrt(D)
    dot_products.masked_fill_(~mask, -torch.inf)

    attn_weights = torch.softmax(dot_products, dim=-1)
    return attn_weights
    

def _ref_causal(Q,K):
    d = Q.size(-1)
    S = torch.einsum("btd,bkd->btk",Q,K)/math.sqrt(d)
    T = S.size(-1)

    mask = torch.triu(torch.ones(T,T,device=Q.device,dtype=torch.bool),1)
    return torch.softmax(S.masked_fill(mask,float("-inf")), -1)

def _self_check():
    Q,K=torch.randn(1,6,4),torch.randn(1,6,4)
    try: out=causal_attention_scores(Q,K)
    except NotImplementedError:
        print("🔧  Implement causal_attention_scores.")
        return
    assert torch.allclose(out,_ref_causal(Q,K),atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 13 – Min–max scale every row
------------------------------------
For X ∈ ℝ^{B×D} return

    Yᵢⱼ = (Xᵢⱼ − min(Xᵢ)) / (max(Xᵢ) − min(Xᵢ) + ε)

so every row is mapped to the range [0, 1].

No Python loops allowed.

In [22]:
import torch

def minmax_rows(X: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    # X: (B, D)
    Xmin, _ = torch.min(X, dim=-1, keepdim=True)
    Xmax, _ = torch.max(X, dim=-1, keepdim=True)

    Y = (X - Xmin) / (Xmax - Xmin + eps)
    return Y


# reference & test
def _ref_minmax_rows(X, eps=1e-8):
    xmin = X.min(dim=1, keepdim=True).values
    xmax = X.max(dim=1, keepdim=True).values
    return (X - xmin) / (xmax - xmin + eps)

def _self_check():
    X = torch.randn(4, 7)
    try:
        out = minmax_rows(X)
    except NotImplementedError:
        print("🔧  Implement minmax_rows.")
        return
    assert torch.allclose(out, _ref_minmax_rows(X), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 14 – Batch RGB→gray conversion
--------------------------------------
Given imgs ∈ ℝ^{N×3×H×W} with values in [0,1], return a grayscale
tensor of shape (N, 1, H, W) using weights

    Y = 0.299 R + 0.587 G + 0.114 B

In [23]:
import torch

def rgb_to_gray(imgs: torch.Tensor) -> torch.Tensor:
    # imgs: B, C, H, W
    w = torch.tensor([0.299, 0.587, 0.114]).view(1, 3, 1, 1)
    return (imgs * w).sum(dim=1, keepdim=True)


def _ref_rgb_to_gray(imgs):
    w = torch.tensor([0.299, 0.587, 0.114], device=imgs.device).view(1,3,1,1)
    return (imgs * w).sum(1, keepdim=True)

def _self_check():
    imgs = torch.rand(2,3,8,8)
    try:
        out = rgb_to_gray(imgs)
    except NotImplementedError:
        print("🔧  Implement rgb_to_gray.")
        return
    assert torch.allclose(out, _ref_rgb_to_gray(imgs), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 15 – σ(z) and σ'(z)
---------------------------
Return both the logistic sigmoid and its derivative for an
arbitrary-shaped tensor z (no loops).

In [24]:
import torch

def sigmoid_and_grad(z: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    s = torch.sigmoid(z)
    return s, s*(1-s)

def _ref_sig_and_grad(z):
    s = torch.sigmoid(z)
    return s, s*(1-s)

def _self_check():
    z = torch.randn(5,3,2)
    try:
        s, ds = sigmoid_and_grad(z)
    except NotImplementedError:
        print("🔧  Implement sigmoid_and_grad.")
        return
    s_ref, ds_ref = _ref_sig_and_grad(z)
    assert torch.allclose(s, s_ref) and torch.allclose(ds, ds_ref)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 16 – Trace & diagonal per batch
---------------------------------------
For M ∈ ℝ^{B×D×D} return

    trace  – shape (B,)
    diag   – shape (B,D)

using only tensor ops.

In [25]:
import torch

def batch_trace_diag(M: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    # M: (B, D, D)
    diag = M.diagonal(dim1=1, dim2=2)
    trace = diag.sum(dim=-1)
    return trace, diag


def _ref_batch_trace_diag(M):
    diag = M.diagonal(dim1=-2, dim2=-1)
    return diag.sum(-1), diag

def _self_check():
    M = torch.randn(3,4,4)
    try:
        tr, d = batch_trace_diag(M)
    except NotImplementedError:
        print("🔧  Implement batch_trace_diag.")
        return
    tr_ref, d_ref = _ref_batch_trace_diag(M)
    assert torch.allclose(tr, tr_ref) and torch.allclose(d, d_ref)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 17 – Label-smoothed CE loss
-----------------------------------
Given logits (B,C), integer targets (B,) and smoothing α ∈ [0,1),
return the scalar loss

    L = − mean_i  sum_c  q_ic  log softmax_i,c

where q is the smoothed one-hot:  
 q_i,target = 1−α, others = α/(C−1).

No loops.

In [26]:
import torch, torch.nn.functional as F

def ls_ce_loss(logits: torch.Tensor, targets: torch.Tensor, alpha: float) -> torch.Tensor:
    B, C = logits.shape
    q = torch.ones_like(logits) * (alpha / (C-1)) # B, C
    q.scatter_(1, targets.view(B, 1), 1-alpha)    # B, C
    return -(q * torch.log_softmax(logits, dim=-1)).sum(dim=-1).mean()

def _ref_ls_ce(logits, targets, alpha):
    B,C = logits.shape
    q = torch.full((B,C), alpha/(C-1), device=logits.device)
    q.scatter_(1, targets.unsqueeze(1), 1-alpha)
    return -(q * F.log_softmax(logits,1)).sum(1).mean()

def _self_check():
    logit = torch.randn(6,10)
    tgt   = torch.randint(0,10,(6,))
    try:
        loss = ls_ce_loss(logit, tgt, 0.1)
    except NotImplementedError:
        print("🔧  Implement ls_ce_loss.")
        return
    assert torch.allclose(loss, _ref_ls_ce(logit,tgt,0.1), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 18 – Jaccard indices
----------------------------
A, B ∈ {0,1}^{B×N×D}.  
Compute J[b, i] = |A[b,0] ∩ B[b,i]| / |A[b,0] ∪ B[b,i]|.

Return J shape (B, N).

In [27]:
import torch

def jaccard_pair(A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
    A0 = A[:, :1, :] # B, 1, D
    inter = (A0 & B).sum(-1)
    union = (A0 | B).sum(-1)
    return inter / (union + 1e-8)


def _ref_jaccard_pair(A,B):
    inter = (A[:,:1] & B).sum(-1)
    union = (A[:,:1] | B).sum(-1)
    return inter / (union + 1e-8)

def _self_check():
    A = (torch.rand(2,1,5)>.5).bool().repeat(1,3,1)
    B = (torch.rand(2,3,5)>.5).bool()
    try:
        out = jaccard_pair(A,B)
    except NotImplementedError:
        print("🔧  Implement jaccard_pair.")
        return
    assert torch.allclose(out, _ref_jaccard_pair(A,B), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 19 – Dice coefficient (mean over classes)
-------------------------------------------------
pred, mask ∈ {0,1}^{N×C×H×W}.  
Return the mean Dice over the C classes:

    Dice_c = (2|∩| + ε) / (|pred_c| + |mask_c| + ε)

In [28]:
import torch

def dice_coeff(pred: torch.Tensor, mask: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    inter = (pred & mask).sum((0,2,3))
    union = pred.sum((0,2,3)) + mask.sum((0,2,3))
    return ((2*inter+eps)/(union+eps)).mean()


def _ref_dice(pred, mask, eps=1e-6):
    inter = (pred & mask).sum((0,2,3))
    union = pred.sum((0,2,3)) + mask.sum((0,2,3))
    return ((2*inter+eps)/(union+eps)).mean()

def _self_check():
    p = (torch.rand(4,3,10,10)>.5)
    m = (torch.rand(4,3,10,10)>.5)
    try:
        d = dice_coeff(p,m)
    except NotImplementedError:
        print("🔧  Implement dice_coeff.")
        return
    assert torch.allclose(d, _ref_dice(p,m), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 20 – Causal conv1d (no bias)
------------------------------------
x: (B,C,T), w: (C_out,C,k).  
Pad (k−1) zeros on the *left* so output has length T.
No loops; use `unfold` or `tensor.unfold`.

In [29]:
import torch

def causal_conv1d(x: torch.Tensor, w: torch.Tensor, dilation: int = 1) -> torch.Tensor:
    B, C_in, T = x.shape
    C_out, C_in, k = w.shape

    x_pad = torch.nn.functional.pad(x, (k-1,0), "constant", 0)
    x_unf = x_pad.unfold(2,k,1)                                 # B, C_in, T, k
    return torch.einsum("bctk,ock->bot", x_unf, w)

def _ref_causal_conv1d(x,w,d=1):
    k = w.size(-1)
    pad = (k-1)*d
    xpad = torch.nn.functional.pad(x,(pad,0))
    x_unf = xpad.unfold(2,k,1).transpose(1,2)  # (B,T,C,k)
    return torch.einsum("btck,ock->bot", x_unf, w)

def _self_check():
    x = torch.randn(1,2,6)
    w = torch.randn(3,2,3)
    try:
        out = causal_conv1d(x,w)
    except NotImplementedError:
        print("🔧  Implement causal_conv1d.")
        return
    assert torch.allclose(out, _ref_causal_conv1d(x,w), atol=1e-5)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 21 – Rolling mean
-------------------------
For x ∈ ℝ^{B×T} and window w (w≤T) return
shape (B, T−w+1) where each entry is the mean of a sliding length-w
window.  Use tensor.unfold; no loops.

In [31]:
import torch

def rolling_mean(x: torch.Tensor, window: int) -> torch.Tensor:
    B, T = x.shape
    x_unf = x.unfold(1,window,1).mean(-1)
    return x_unf

def _ref_rolling_mean(x, w):
    return x.unfold(1,w,1).mean(-1)

def _self_check():
    a = torch.arange(6).float().view(1,-1)
    try:
        out = rolling_mean(a,3)
    except NotImplementedError:
        print("🔧  Implement rolling_mean.")
        return
    assert torch.allclose(out, _ref_rolling_mean(a,3))
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 22 – Gather (B, T, D) at time indices
---------------------------------------------
seq: (B,T,D), idx: (B,K) → out: (B,K,D) using `tensor.gather`.

In [32]:
import torch

def batch_gather(seq: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
    B, T, D = seq.shape
    B, K = idx.shape
    idx = idx.view(B, K, 1).expand(B, K, D)
    return torch.gather(seq, 1, idx)


def _ref_batch_gather(seq, idx):
    return seq.gather(1, idx.unsqueeze(-1).expand(-1,-1,seq.size(-1)))

def _self_check():
    s = torch.randn(2,5,4)
    i = torch.tensor([[1,3],[0,4]])
    try:
        out = batch_gather(s,i)
    except NotImplementedError:
        print("🔧  Implement batch_gather.")
        return
    assert torch.allclose(out, _ref_batch_gather(s,i))
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed
